# EP2: fine-tuning do BERTimbau large (`neuralmind/bert-large-portuguese-cased`)

Mesmo pipeline do BERTimbau base do EP1 (`finetune_bertimbau.py`): truncamento início+fim (128 tokens do início e o
restante do fim, até 512), lotes com padding dinâmico, fp16, AdamW com warmup linear e recorte de gradiente.

**Modos** (variável `MODOS` na célula de configuração):
- `holdout`: treina nos 80% e prevê os 20% de avaliação, os mesmos usados nos clássicos do EP2
  (`divisao_80_20.csv`, fold 1 do GroupKFold agrupado por texto). Serve para comparar com os outros modelos e
  testar o stacking. Grava as probabilidades por época.
- `final`: treina em 100% do `train.xlsx` e grava as probabilidades do `test1.xlsx` (sem rótulos) e do `test2.xlsx`.

**Como rodar no Kaggle**
1. Crie um Dataset com os 4 arquivos da pasta `dados/` (train.xlsx, test1.xlsx, test2.xlsx, divisao_80_20.csv).
2. Importe este notebook e adicione o Dataset (Add Input).
3. Settings: Accelerator = **GPU T4** (não use a P100: as versões atuais do PyTorch não a suportam) e Internet = On
   (para baixar o modelo).
4. **Save Version > Save & Run All (Commit)**: roda em segundo plano, sem precisar deixar o navegador aberto
   (limite de 12 h). Ao terminar, baixe os arquivos em Output (`/kaggle/working/saida`).

**Colab**: Ambiente de execução > GPU T4; envie os 4 arquivos para `/content` (ou monte o Drive e ajuste
`PASTA_DADOS`). A saída fica em `/content/saida`.

In [ ]:
# ===================== Configuração =====================
MODOS = ['holdout']            # 'holdout', 'final' ou os dois: ['holdout', 'final'] (~7-8 h no total na T4)
MODELO = 'neuralmind/bert-large-portuguese-cased'
MAX_LEN = 512                  # tamanho máximo em tokens
TOKENS_INICIO = 128            # tokens mantidos do início em textos longos; o restante vem do fim
EPOCAS = 2
LR = 1.5e-5                    # o base do EP1 usou 2e-5; o large costuma ser mais instável com lr alto
LOTE = 8                       # tamanho do lote na GPU
ACUMULACAO = 2                 # lote efetivo = LOTE x ACUMULACAO = 16 (igual ao EP1)
WARMUP = 0.1                   # fração dos passos com aquecimento do lr
SEMENTE = 42
GRADIENT_CHECKPOINTING = False # True economiza memória (mais lento); use se der "CUDA out of memory"
SALVAR_MODELO_FINAL = False    # True grava o modelo final (~1,3 GB) em saida/modelo_final
PASTA_DADOS = None             # None = procura train.xlsx em /kaggle/input, /content e na pasta atual

In [ ]:
import glob
import json
import math
import os
import random
import re
import time

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer

CLASSES = ['c1', 'c234', 'c5']
ROTULO_PARA_ID = {c: i for i, c in enumerate(CLASSES)}

if PASTA_DADOS is None:
    achados = (glob.glob('/kaggle/input/**/train.xlsx', recursive=True) + glob.glob('/content/**/train.xlsx', recursive=True)
               + glob.glob('train.xlsx'))
    assert achados, 'train.xlsx não encontrado: adicione o Dataset (Kaggle) ou envie os arquivos (Colab)'
    PASTA_DADOS = os.path.dirname(achados[0])
SAIDA = '/kaggle/working/saida' if os.path.exists('/kaggle/working') else os.path.abspath('saida')
os.makedirs(SAIDA, exist_ok=True)

dispositivo = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert dispositivo.type == 'cuda', 'sem GPU: ative o acelerador (T4)'
print('Dados:', PASTA_DADOS, '| Saída:', SAIDA)
print('GPU:', torch.cuda.get_device_name(0), f'({torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GB)')
print('torch', torch.__version__)

treino = pd.read_excel(os.path.join(PASTA_DADOS, 'train.xlsx'))
treino['resp_text'] = treino['resp_text'].fillna('').astype(str)
divisao = pd.read_csv(os.path.join(PASTA_DADOS, 'divisao_80_20.csv'))
assert len(divisao) == len(treino) and (divisao['indice'].to_numpy() == np.arange(len(treino))).all()
print(treino.shape, treino['clarity'].value_counts().to_dict(), divisao['conjunto'].value_counts().to_dict())

In [ ]:
def chave_texto(texto):
    """Normalização do EP1: textos que só diferem em espaços/quebras de linha são o mesmo texto."""
    return re.sub(r'\s+', ' ', texto).strip()


def fixar_semente(semente):
    random.seed(semente)
    np.random.seed(semente)
    torch.manual_seed(semente)
    torch.cuda.manual_seed_all(semente)


def tokenizar(textos, tokenizer, max_len, tokens_inicio):
    """Tokeniza os textos; os que passam de max_len mantêm o início e o fim."""
    corpo = max_len - 2  # espaço para [CLS] e [SEP]
    tokens_fim = corpo - tokens_inicio
    textos = [chave_texto(str(t)) for t in textos]
    codificados = tokenizer(textos, add_special_tokens=False, truncation=False)['input_ids']
    ids = []
    for tokens in codificados:
        if len(tokens) > corpo:
            tokens = tokens[:tokens_inicio] + (tokens[-tokens_fim:] if tokens_fim > 0 else [])
        ids.append([tokenizer.cls_token_id] + tokens + [tokenizer.sep_token_id])
    return ids


def criar_lotes(ids, tamanho_lote, embaralhar, rng=None):
    """Agrupa exemplos de tamanho parecido no mesmo lote (menos padding = treino mais rápido)."""
    indices = np.arange(len(ids))
    if embaralhar:
        rng.shuffle(indices)
        tamanho_bloco = tamanho_lote * 50
        blocos = [indices[i:i + tamanho_bloco] for i in range(0, len(indices), tamanho_bloco)]
        indices = np.concatenate([sorted(b, key=lambda i: len(ids[i])) for b in blocos])
    else:
        indices = np.array(sorted(indices, key=lambda i: len(ids[i])))
    lotes = [indices[i:i + tamanho_lote] for i in range(0, len(indices), tamanho_lote)]
    if embaralhar:
        rng.shuffle(lotes)
    return lotes


def montar_lote(ids, lote, id_padding):
    maior = max(len(ids[i]) for i in lote)
    input_ids = torch.full((len(lote), maior), id_padding, dtype=torch.long)
    mascara = torch.zeros((len(lote), maior), dtype=torch.long)
    for j, i in enumerate(lote):
        n = len(ids[i])
        input_ids[j, :n] = torch.tensor(ids[i], dtype=torch.long)
        mascara[j, :n] = 1
    return input_ids.to(dispositivo), mascara.to(dispositivo)


def criar_otimizador(modelo, lr):
    sem_decay = ['bias', 'LayerNorm.weight']
    grupos = [
        {'params': [p for n, p in modelo.named_parameters() if not any(s in n for s in sem_decay)], 'weight_decay': 0.01},
        {'params': [p for n, p in modelo.named_parameters() if any(s in n for s in sem_decay)], 'weight_decay': 0.0},
    ]
    return torch.optim.AdamW(grupos, lr=lr)


def criar_agendador(otimizador, passos_totais, fracao_warmup):
    passos_warmup = int(passos_totais * fracao_warmup)

    def fator(passo):
        if passo < passos_warmup:
            return passo / max(1, passos_warmup)
        return max(0.0, (passos_totais - passo) / max(1, passos_totais - passos_warmup))

    return torch.optim.lr_scheduler.LambdaLR(otimizador, fator)


@torch.no_grad()
def prever_probabilidades(modelo, ids, id_padding):
    """Matriz de probabilidades (n x 3) na ordem original dos textos."""
    modelo.eval()
    probs = np.zeros((len(ids), len(CLASSES)), dtype=np.float32)
    for lote in criar_lotes(ids, LOTE * 4, embaralhar=False):
        input_ids, mascara = montar_lote(ids, lote, id_padding)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = modelo(input_ids=input_ids, attention_mask=mascara).logits
        probs[lote] = torch.softmax(logits.float(), dim=-1).cpu().numpy()
    return probs


def salvar_probs(probs, caminho, rotulos=None, indices=None):
    df = pd.DataFrame(probs, columns=[f'p_{c}' for c in CLASSES])
    df.insert(0, 'indice', np.arange(len(probs)) if indices is None else indices)
    if rotulos is not None:
        df['rotulo'] = rotulos
    df['previsto'] = np.array(CLASSES)[probs.argmax(1)]
    df.to_csv(caminho, index=False)


def treinar(ids_treino, y_treino, id_padding, ids_val=None, y_val=None, ao_fim_da_epoca=None):
    fixar_semente(SEMENTE)
    rng = np.random.default_rng(SEMENTE)
    modelo = AutoModelForSequenceClassification.from_pretrained(
        MODELO, num_labels=len(CLASSES), id2label=dict(enumerate(CLASSES)), label2id=ROTULO_PARA_ID,
    ).float().to(dispositivo)
    if GRADIENT_CHECKPOINTING:
        modelo.gradient_checkpointing_enable()

    otimizador = criar_otimizador(modelo, LR)
    lotes_por_epoca = math.ceil(len(ids_treino) / LOTE)
    passos_totais = math.ceil(lotes_por_epoca / ACUMULACAO) * EPOCAS
    agendador = criar_agendador(otimizador, passos_totais, WARMUP)
    escalador = torch.amp.GradScaler('cuda')
    y_treino = torch.tensor(y_treino, dtype=torch.long)

    historico = []
    for epoca in range(1, EPOCAS + 1):
        modelo.train()
        inicio, perda_acumulada = time.time(), 0.0
        lotes = criar_lotes(ids_treino, LOTE, embaralhar=True, rng=rng)
        otimizador.zero_grad()
        for n_lote, lote in enumerate(lotes, start=1):
            input_ids, mascara = montar_lote(ids_treino, lote, id_padding)
            rotulos = y_treino[torch.as_tensor(lote)].to(dispositivo)
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits = modelo(input_ids=input_ids, attention_mask=mascara).logits
            perda = F.cross_entropy(logits.float(), rotulos)
            escalador.scale(perda / ACUMULACAO).backward()
            perda_acumulada += perda.item()
            if n_lote % ACUMULACAO == 0 or n_lote == len(lotes):
                escalador.unscale_(otimizador)
                torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
                escalador.step(otimizador)
                escalador.update()
                otimizador.zero_grad()
                agendador.step()
            if n_lote % 200 == 0:
                decorrido = time.time() - inicio
                print(f'  época {epoca} | lote {n_lote}/{len(lotes)} | perda {perda_acumulada / n_lote:.4f} '
                      f'| ~{decorrido / n_lote * (len(lotes) - n_lote) / 60:.1f} min restantes', flush=True)

        registro = {'epoca': epoca, 'perda_treino': perda_acumulada / len(lotes), 'minutos': (time.time() - inicio) / 60}
        if ids_val is not None:
            probs_val = prever_probabilidades(modelo, ids_val, id_padding)
            previsto = probs_val.argmax(1)
            registro['acuracia'] = float((previsto == y_val).mean())
            registro['distribuicao_prevista'] = str(np.bincount(previsto, minlength=3).tolist())
            if ao_fim_da_epoca is not None:
                ao_fim_da_epoca(epoca, probs_val)
        historico.append(registro)
        print('  ' + ' | '.join(f'{k}={v:.4f}' if isinstance(v, float) else f'{k}={v}' for k, v in registro.items()),
              flush=True)
    return modelo, historico

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODELO)
config = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (str, int, float, bool, list, type(None)))}
with open(os.path.join(SAIDA, 'config.json'), 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

ids_todos = tokenizar(treino['resp_text'], tokenizer, MAX_LEN, TOKENS_INICIO)
y_todos = treino['clarity'].map(ROTULO_PARA_ID).to_numpy()
tamanhos = np.array([len(i) for i in ids_todos])
print(f'Tokens por texto: mediana {np.median(tamanhos):.0f}, p90 {np.percentile(tamanhos, 90):.0f}, '
      f'{(tamanhos >= MAX_LEN).mean():.1%} truncados')

if 'holdout' in MODOS:
    print('\n========== HOLDOUT 80/20 ==========')
    idx_tr = np.where(divisao['conjunto'] == 'treino')[0]
    idx_av = np.where(divisao['conjunto'] == 'avaliacao')[0]
    ids_tr, ids_av = [ids_todos[i] for i in idx_tr], [ids_todos[i] for i in idx_av]

    def ao_fim_da_epoca(epoca, probs):
        salvar_probs(probs, os.path.join(SAIDA, f'holdout_probabilidades_epoca_{epoca}.csv'),
                     rotulos=treino['clarity'].to_numpy()[idx_av], indices=idx_av)

    modelo, historico = treinar(ids_tr, y_todos[idx_tr], tokenizer.pad_token_id, ids_av, y_todos[idx_av], ao_fim_da_epoca)
    pd.DataFrame(historico).to_csv(os.path.join(SAIDA, 'holdout_historico.csv'), index=False)
    del modelo
    torch.cuda.empty_cache()

if 'final' in MODOS:
    print('\n========== FINAL (100% do treino) ==========')
    modelo, historico = treinar(ids_todos, y_todos, tokenizer.pad_token_id)
    pd.DataFrame(historico).to_csv(os.path.join(SAIDA, 'final_historico.csv'), index=False)
    for nome in ['test1', 'test2']:
        teste = pd.read_excel(os.path.join(PASTA_DADOS, f'{nome}.xlsx'))
        ids_teste = tokenizar(teste['resp_text'].fillna('').astype(str), tokenizer, MAX_LEN, TOKENS_INICIO)
        probs = prever_probabilidades(modelo, ids_teste, tokenizer.pad_token_id)
        salvar_probs(probs, os.path.join(SAIDA, f'final_probabilidades_{nome}.csv'))
        print(nome, len(teste), 'textos | distribuição prevista:',
              pd.Series(np.array(CLASSES)[probs.argmax(1)]).value_counts().to_dict())
    if SALVAR_MODELO_FINAL:
        modelo.save_pretrained(os.path.join(SAIDA, 'modelo_final'))
        tokenizer.save_pretrained(os.path.join(SAIDA, 'modelo_final'))

print('\nArquivos gerados:', sorted(os.listdir(SAIDA)))